# Phase 3: Final Cleaned Dataset Pipeline

This notebook combines all three source datasets and applies cleaning in one run.

Rules applied:
- aggressive dedupe (exact + near duplicate)
- noise filtering (very short, URL-only, emoji/symbol-only, repeated-char spam)
- Sinhala-priority filtering
- long-comment exclusion threshold

In [3]:
from pathlib import Path
import subprocess
import pandas as pd

REPO_ROOT = Path.cwd().resolve()
for p in [REPO_ROOT] + list(REPO_ROOT.parents):
    if (p / 'data_collection').exists():
        REPO_ROOT = p
        break
DATA_ROOT = Path('/root/separate_volume')
if not DATA_ROOT.exists():
    DATA_ROOT = REPO_ROOT
RUN_ID = pd.Timestamp.now().strftime('preprocess_%Y%m%d_%H%M%S')

output_csv = DATA_ROOT / 'datasets/preprocessing/runs' / RUN_ID / 'final_cleaned_dataset.csv'
summary_path = DATA_ROOT / 'datasets/preprocessing/runs' / RUN_ID / 'final_cleaned_dataset_summary.json'

params = {
    'sinhala_threshold': 0.2,
    'min_text_chars': 8,
    'max_text_chars': 1200,
}
output_csv, summary_path, params


(WindowsPath('D:/client-projects/sl-social-media-risk-analysis/datasets/preprocessing/runs/preprocess_20260316_231039/final_cleaned_dataset.csv'),
 WindowsPath('D:/client-projects/sl-social-media-risk-analysis/datasets/preprocessing/runs/preprocess_20260316_231039/final_cleaned_dataset_summary.json'),
 {'sinhala_threshold': 0.2, 'min_text_chars': 8, 'max_text_chars': 1200})

In [4]:
elakiri_path = DATA_ROOT / 'datasets/sources/elakiri_comments.csv'
gossip_path = DATA_ROOT / 'datasets/sources/gossip_lanka_comments.csv'
youtube_path = DATA_ROOT / 'datasets/sources/youtube_comments.csv'

cmd = [
    'python', '-m', 'data_collection.pipelines.build_final_cleaned_dataset',
    '--elakiri-path', str(elakiri_path),
    '--gossip-path', str(gossip_path),
    '--youtube-path', str(youtube_path),
    '--output-csv', str(output_csv),
    '--summary-path', str(summary_path),
    '--sinhala-threshold', str(params['sinhala_threshold']),
    '--min-text-chars', str(params['min_text_chars']),
    '--max-text-chars', str(params['max_text_chars']),
]

print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(REPO_ROOT))


Running: python -m data_collection.pipelines.build_final_cleaned_dataset --output-csv D:\client-projects\sl-social-media-risk-analysis\datasets\preprocessing\runs\preprocess_20260316_231039\final_cleaned_dataset.csv --summary-path D:\client-projects\sl-social-media-risk-analysis\datasets\preprocessing\runs\preprocess_20260316_231039\final_cleaned_dataset_summary.json --sinhala-threshold 0.2 --min-text-chars 8 --max-text-chars 1200


CompletedProcess(args=['python', '-m', 'data_collection.pipelines.build_final_cleaned_dataset', '--output-csv', 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\preprocessing\\runs\\preprocess_20260316_231039\\final_cleaned_dataset.csv', '--summary-path', 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\preprocessing\\runs\\preprocess_20260316_231039\\final_cleaned_dataset_summary.json', '--sinhala-threshold', '0.2', '--min-text-chars', '8', '--max-text-chars', '1200'], returncode=0)

In [5]:
import json
summary = json.loads(summary_path.read_text(encoding="utf-8"))
summary

{'created_at': '2026-03-16T17:41:50.473809+00:00',
 'output_csv': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\preprocessing\\runs\\preprocess_20260316_231039\\final_cleaned_dataset.csv',
 'summary_path': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\preprocessing\\runs\\preprocess_20260316_231039\\final_cleaned_dataset_summary.json',
 'sources': {'elakiri': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\sources\\elakiri_comments.csv',
  'gossip_lanka': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\sources\\gossip_lanka_comments.csv',
  'youtube': 'D:\\client-projects\\sl-social-media-risk-analysis\\datasets\\sources\\youtube_comments.csv'},
 'constraints': {'sinhala_threshold': 0.2,
  'min_text_chars': 8,
  'max_text_chars': 1200},
 'stats': {'input_rows': 75613,
  'kept_rows': 73691,
  'dropped_non_sinhala': 0,
  'dropped_long_rows': 145,
  'dropped_noise': 651,
  'dropped_exact_duplicates': 69,
  'dropped_near_dupli